# Projekt 03 (final): Spamfilter mit Naive Bayes — auf echten Daten

**Ziel:** Du baust einen kompletten Spam-Klassifikator **von Grund auf selbst** —
mit nichts als der Bayes-Regel, Zaehlen und Logarithmen — und misst ihn an echten
Daten: 5.574 echten SMS (davon ~13 % Spam) aus der *SMS Spam Collection* (UCI).
Am Ende vergleichst du dein Modell mit der scikit-learn-Implementierung.

**Vorbereitung:** Einmalig im Terminal (aus dem Ordner `03-final`, venv aktiv):

```
python datasets/download_data.py
```

**Bezug zum Skript:** Abschnitt 2.4 (Bayes-Regel, Naive Bayes) und 2.5 (Supervised Learning).

## 1. Daten laden und anschauen

Regel Nummer eins bei echten Daten: **erst anschauen, dann modellieren**.

In [1]:
import os
import pandas as pd

# Funktioniert auch, wenn das Notebook aus dem solution/-Ordner gestartet wird:
DATEN = "datasets/SMSSpamCollection"
if not os.path.exists(DATEN):
    DATEN = "../" + DATEN

df = pd.read_csv(DATEN, sep="\t", header=None,
                 names=["label", "text"], quoting=3)  # quoting=3: " nicht als Anfuehrungszeichen deuten
print(df.shape)
df.head()

(5574, 2)


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [2]:
print(df["label"].value_counts())
print(f"\nSpam-Anteil: {(df['label'] == 'spam').mean():.1%}")

# Sind Spam-SMS laenger? (Typisch: ja, deutlich)
df["laenge"] = df["text"].str.len()
df.groupby("label")["laenge"].describe().round(1)

label
ham     4827
spam     747
Name: count, dtype: int64

Spam-Anteil: 13.4%


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4827.0,71.5,58.3,2.0,33.0,52.0,93.0,910.0
spam,747.0,138.7,28.9,13.0,133.0,149.0,157.0,223.0


**Wichtig — unausgewogene Klassen:** Nur ~13 % Spam. Ein "Klassifikator", der stur
*ham* sagt, haette schon ~87 % Accuracy! Accuracy allein reicht hier also nicht als
Erfolgsmass — das pruefen wir spaeter mit Precision und Recall nach.

## 2. Trainings- und Testmenge

Wir bewerten das Modell nur auf SMS, die es beim Training **nie gesehen** hat
(Skript 2.5: Generalisierung!). `stratify` sorgt dafuer, dass der Spam-Anteil in
beiden Teilmengen gleich ist; der feste `random_state` macht alles reproduzierbar.

In [3]:
from sklearn.model_selection import train_test_split

train_texte, test_texte, train_labels, test_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"])

print(f"Training: {len(train_texte)} SMS, Test: {len(test_texte)} SMS")
print(f"Spam-Anteil Training: {(train_labels == 'spam').mean():.1%}, "
      f"Test: {(test_labels == 'spam').mean():.1%}")

Training: 4459 SMS, Test: 1115 SMS
Spam-Anteil Training: 13.4%, Test: 13.4%


## 3. Vom Text zu Woertern: Tokenisierung

Naive Bayes arbeitet mit Wort-Wahrscheinlichkeiten — also muessen wir SMS in
Woerter zerlegen. Wir halten es bewusst einfach: Kleinbuchstaben, dann alle
Folgen von Buchstaben/Ziffern extrahieren.

**Aufgabe:** Implementiere `tokenisiere(text)` mit dem vorbereiteten Regex.

In [4]:
import re

WORT_MUSTER = re.compile(r"[a-z0-9']+")

def tokenisiere(text):
    """Zerlegt einen Text in eine Liste kleingeschriebener Woerter."""
    return WORT_MUSTER.findall(text.lower())

# Mini-Test — muss True ausgeben:
print(tokenisiere("WINNER!! Claim your £900 prize now!") == ["winner", "claim", "your", "900", "prize", "now"])

True


## 4. Naive Bayes von Grund auf

Zur Erinnerung (Skript 2.4): Fuer eine SMS mit Woertern $w_1, \dots, w_n$ vergleichen wir

$$P(\text{spam} \mid w_1..w_n) \propto P(\text{spam}) \prod_i P(w_i \mid \text{spam})
\qquad \text{vs.} \qquad
P(\text{ham} \mid w_1..w_n) \propto P(\text{ham}) \prod_i P(w_i \mid \text{ham})$$

Zwei Praxis-Tricks, beide im Code vorbereitet:

1. **Log-Wahrscheinlichkeiten:** Das Produkt aus hunderten kleinen Zahlen wuerde
   numerisch zu 0 kollabieren (*underflow*). Darum rechnen wir mit Summen von
   Logarithmen: $\log P(c) + \sum_i \log P(w_i \mid c)$ — der Vergleich bleibt derselbe,
   weil der Logarithmus monoton ist.
2. **Laplace-Glaettung:** Ein Wort, das im Training nie in Spam vorkam, haette
   $P(w \mid \text{spam}) = 0$ — ein einziges solches Wort wuerde jede Spam-SMS
   "freisprechen" ($\log 0 = -\infty$). Darum tun wir so, als haetten wir jedes
   bekannte Wort in jeder Klasse **einmal extra** gesehen ($\alpha = 1$):

$$P(w \mid c) = \frac{\text{Anzahl}(w, c) + 1}{\text{Woerter gesamt in } c + |V|}$$

wobei $|V|$ die Groesse des Vokabulars (alle bekannten Woerter) ist.

**Aufgabe:** Vervollstaendige `trainiere` und `log_posterior`.

In [5]:
from collections import Counter
import math

def trainiere(texte, labels):
    """Zaehlt alles, was Naive Bayes braucht. Liefert ein Modell-Dictionary."""
    anzahl_sms = Counter(labels)                       # {"ham": ..., "spam": ...}
    wortzahl = {"ham": Counter(), "spam": Counter()}   # Wort -> Anzahl je Klasse
    for text, label in zip(texte, labels):
        wortzahl[label].update(tokenisiere(text))
    vokabular = set(wortzahl["ham"]) | set(wortzahl["spam"])
    return {
        "log_prior": {c: math.log(anzahl_sms[c] / len(labels)) for c in ("ham", "spam")},
        "wortzahl": wortzahl,
        "gesamt": {c: sum(wortzahl[c].values()) for c in ("ham", "spam")},
        "V": len(vokabular),
        "vokabular": vokabular,
    }

def log_wort_wkt(modell, wort, klasse):
    """log P(wort | klasse) mit Laplace-Glaettung (alpha = 1)."""
    zaehler = modell["wortzahl"][klasse][wort] + 1
    nenner = modell["gesamt"][klasse] + modell["V"]
    return math.log(zaehler / nenner)

def log_posterior(modell, text, klasse):
    """log( P(klasse) * prod P(w|klasse) ) — unbekannte Woerter werden ignoriert."""
    ergebnis = modell["log_prior"][klasse]
    for wort in tokenisiere(text):
        if wort in modell["vokabular"]:
            ergebnis += log_wort_wkt(modell, wort, klasse)
    return ergebnis

def klassifiziere(modell, text):
    return "spam" if log_posterior(modell, text, "spam") > log_posterior(modell, text, "ham") else "ham"

modell = trainiere(train_texte, train_labels)
print(f"Vokabular: {modell['V']} Woerter")
print(klassifiziere(modell, "URGENT! You have won a free prize, call now!"))   # erwartet: spam
print(klassifiziere(modell, "Ok, see you at the station at 6"))               # erwartet: ham

Vokabular: 7899 Woerter
spam
ham


## 5. Wie gut ist der Filter wirklich?

Jetzt die Bewaehrungsprobe auf den zurueckgehaltenen Testdaten. Neben der
Accuracy schauen wir auf die **Confusion Matrix** und zwei Kennzahlen:

- **Precision** (Spam): Wenn der Filter "Spam" sagt — wie oft stimmt das?
  *(Wichtig: Falsch-Positive = echte SMS im Spamordner = sehr aergerlich!)*
- **Recall** (Spam): Wie viel vom echten Spam faengt der Filter?

In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

vorhersagen = [klassifiziere(modell, t) for t in test_texte]

print(f"Accuracy: {accuracy_score(test_labels, vorhersagen):.4f}")
print(f"(Zum Vergleich — stur 'ham' sagen: {(test_labels == 'ham').mean():.4f})\n")
print(pd.DataFrame(confusion_matrix(test_labels, vorhersagen, labels=["ham", "spam"]),
                   index=["echt: ham", "echt: spam"], columns=["vorhergesagt: ham", "vorhergesagt: spam"]))
print()
print(classification_report(test_labels, vorhersagen, digits=3))

Accuracy: 0.9857
(Zum Vergleich — stur 'ham' sagen: 0.8664)

            vorhergesagt: ham  vorhergesagt: spam
echt: ham                 964                   2
echt: spam                 14                 135

              precision    recall  f1-score   support

         ham      0.986     0.998     0.992       966
        spam      0.985     0.906     0.944       149

    accuracy                          0.986      1115
   macro avg      0.986     0.952     0.968      1115
weighted avg      0.986     0.986     0.985      1115



## 6. Was hat das Modell gelernt?

Ein grosser Vorteil von Naive Bayes: Man kann **hineinschauen**. Welche Woerter
sprechen am staerksten fuer Spam? Wir ranken per Log-Verhaeltnis
$\log \frac{P(w \mid \text{spam})}{P(w \mid \text{ham})}$ (nur Woerter, die mindestens 5-mal vorkommen).

In [7]:
def spam_indiz(wort):
    return log_wort_wkt(modell, wort, "spam") - log_wort_wkt(modell, wort, "ham")

haeufig = [w for w in modell["vokabular"]
           if modell["wortzahl"]["spam"][w] + modell["wortzahl"]["ham"][w] >= 5]
top_spam = sorted(haeufig, key=spam_indiz, reverse=True)[:15]
top_ham = sorted(haeufig, key=spam_indiz)[:15]
print("Staerkste Spam-Woerter:", ", ".join(top_spam))
print("\nStaerkste Ham-Woerter: ", ", ".join(top_ham))

Staerkste Spam-Woerter: claim, prize, won, 150p, tone, 18, guaranteed, cs, 500, awarded, 1000, landline, 150ppm, www, uk

Staerkste Ham-Woerter:  gt, lt, he, i'll, da, lor, later, she, amp, ask, anything, doing, cos, home, said


## 7. Vergleich mit scikit-learn

Zum Abschluss dasselbe Modell mit den Standardwerkzeugen der Praxis:
`CountVectorizer` (zaehlt Woerter) + `MultinomialNB` (genau unser Algorithmus).
Wenn deine Von-Hand-Version gut ist, liegen beide nah beieinander.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

vektorisierer = CountVectorizer(token_pattern=r"[a-z0-9']+", lowercase=True)
X_train = vektorisierer.fit_transform(train_texte)
X_test = vektorisierer.transform(test_texte)

sk_modell = MultinomialNB(alpha=1.0)
sk_modell.fit(X_train, train_labels)
sk_accuracy = sk_modell.score(X_test, test_labels)

meine_accuracy = accuracy_score(test_labels, vorhersagen)
print(f"Mein Naive Bayes:        {meine_accuracy:.4f}")
print(f"scikit-learn Multinomial: {sk_accuracy:.4f}")

Mein Naive Bayes:        0.9857
scikit-learn Multinomial: 0.9857


*(Kleine Abweichungen sind normal: scikit-learn multipliziert mehrfach vorkommende
Woerter pro SMS mit, waehrend Details wie der Umgang mit unbekannten Woertern bei
uns minimal anders geloest sind.)*

## Geschafft — was du jetzt kannst

- einen echten Datensatz laden, explorieren und sauber in Train/Test teilen
- die Bayes-Regel in einen funktionierenden Klassifikator uebersetzen
  (inkl. der zwei Praxis-Tricks: Log-Raum und Laplace-Glaettung)
- ein Modell mit den *richtigen* Metriken bewerten (Precision/Recall statt nur Accuracy)
- dein Von-Hand-Modell gegen eine Industrie-Implementierung benchmarken

**Bonusaufgaben** (optional, ohne Musterloesung):
1. Der Filter steckt manche echte SMS in den Spamordner (Falsch-Positive). Schau dir
   diese SMS an (`test_texte[(vorhersagen_series == "spam") & (test_labels == "ham")]`) — warum stolpert das Modell?
2. Experimentiere mit der Glaettung: Was passiert bei $\alpha = 0{,}01$ oder $\alpha = 10$?
3. Nimm Woerter, die nur 1-mal vorkommen, aus dem Vokabular. Wird das Modell besser oder schlechter — und warum koennte beides passieren?